[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/en_clase/S08_02_random_forest_en_clase.ipynb)

# S08 · 02 — Random Forest: la sabiduría de muchos árboles · versión en clase

⏱ 30 min en clase · Sprint 8 · M3 (UD 3.7)

**Contexto TurisData.** El árbol podado nos dio un AUC de ≈ 0,72 y era muy inestable. Si en lugar de **un** árbol preguntamos a **cientos** de árboles distintos y hacemos votar, los errores individuales se compensan. Eso es un **bosque aleatorio** (*Random Forest*).

**Este cuaderno tiene 2 ejercicios** (1 a 2), los que se hacen en el aula. La importancia por permutación, el coste del bosque, el mini-reto y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1 y 2 son los 2 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])
Xtr = prep.fit_transform(X_train)
Xte = prep.transform(X_test)
nombres = prep.get_feature_names_out()
print("Listo ✔")

### 👀 A2 · EJEMPLO — Qué reservas se quedan fuera de una muestra bootstrap

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué porcentaje de reservas no entra en una muestra bootstrap?
2. ¿Qué quiere decir muestreo «con reemplazo»?
3. ¿Para qué sirven esas reservas que quedan fuera?

In [ ]:
# Cuántas reservas quedan fuera de una muestra bootstrap: ≈ 36,8 %
rng = np.random.default_rng(SEMILLA)
n = 5000
fuera = len(set(range(n)) - set(rng.integers(0, n, n))) / n
print(f"Reservas que NO entran en una muestra bootstrap: {fuera:.1%}")

### ✏️ A3 · EJERCICIO 1 de 2 — Bosque de 300 árboles frente a un árbol podado

**Tu turno:** Entrena `bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)` con train. Guarda `auc_test_bosque` (AUC en test con `predict_proba`) y el AUC en test del árbol podado de la práctica anterior (`DecisionTreeClassifier(min_samples_leaf=80, random_state=SEMILLA)`) en `auc_test_arbol`.

**Debe salir:** `Árbol podado: 0.716 · Bosque de 300 árboles: 0.734` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué AUC tienen el árbol podado y el bosque?
2. ¿Cuánto mejora el bosque al árbol?
3. ¿Por qué el bosque mejora a un solo árbol si cada árbol del bosque es peor que uno podado?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: fit sobre Xtr, y_train; roc_auc_score(y_test, modelo.predict_proba(Xte)[:, 1]) para ambos
bosque = ...
auc_test_bosque = ...
arbol = ...
auc_test_arbol = ...

assert auc_test_bosque > auc_test_arbol, "El bosque debería superar a un único árbol podado"
assert auc_test_bosque > 0.70, "El AUC del bosque debería superar 0,70"
print(f"Árbol podado: {auc_test_arbol:.3f} · Bosque de 300 árboles: {auc_test_bosque:.3f}")
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — El AUC out-of-bag según el número de árboles

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué AUC OOB tiene el bosque con 5 árboles y con 400?
2. ¿Cuándo empieza a aplanarse la curva?
3. ¿Qué ventaja tiene la puntuación OOB frente a la validación cruzada?

In [ ]:
n_arboles = [5, 10, 25, 50, 100, 200, 400]
oob_auc = []
for k in n_arboles:
    b = RandomForestClassifier(n_estimators=k, min_samples_leaf=5, oob_score=True,
                               random_state=SEMILLA, n_jobs=2).fit(Xtr, y_train)
    # oob_decision_function_ contiene la probabilidad OOB de cada reserva de train; con pocos árboles, algunas
    # reservas no quedan fuera en ninguno (NaN) y se rellenan con 0,5 («no sé»)
    oob_auc.append(roc_auc_score(y_train, np.nan_to_num(b.oob_decision_function_[:, 1], nan=0.5)))
plt.figure(figsize=(7, 3.8))
plt.plot(n_arboles, oob_auc, marker="o", color="#2a6f97")
plt.xscale("log")
plt.title("Random Forest: AUC out-of-bag según el número de árboles")
plt.xlabel("Número de árboles (escala logarítmica)")
plt.ylabel("AUC out-of-bag")
plt.show()

### ✏️ A5 · EJERCICIO 2 de 2 — Rendimientos decrecientes

**Tu turno:** Con la lista `oob_auc`, guarda en `mejora_5_a_400` cuánto sube el AUC OOB al pasar de 5 a 400 árboles (`oob_auc[-1] - oob_auc[0]`) y en `mejora_100_a_400` lo que sube de 100 a 400 árboles. Fíjate en que la segunda es mucho menor: los rendimientos son decrecientes.

**Debe salir:** `5 → 400 árboles: +0.103 · 100 → 400 árboles: +0.005` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto vale `mejora_5_a_400` y cuánto `mejora_100_a_400`?
2. ¿Cuántas veces mayor es la primera mejora?
3. ¿Qué recomiendas: usar 100 o 400 árboles?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: los índices de n_arboles: 5 está en la posición 0, 100 en la 4 y 400 en la última
mejora_5_a_400 = ...
mejora_100_a_400 = ...

assert mejora_5_a_400 > 0.05, "De 5 a 400 árboles el AUC debería subir bastante"
assert abs(mejora_100_a_400) < mejora_5_a_400 / 2, "De 100 a 400 la mejora es mucho menor (se satura)"
print(f"5 → 400 árboles: +{mejora_5_a_400:.3f} · 100 → 400 árboles: {mejora_100_a_400:+.3f}")
print("✔ Ejercicio 2 correcto")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/para_casa/S08_02_random_forest.ipynb)** trae la importancia por permutación, el coste del bosque frente al árbol, el mini-reto del duelo con la regresión logística y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, el árbol, el bosque aleatorio, la importancia por permutación, la regresión logística y el preprocesado.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `reservas_turisdata.csv` (6000 reservas), crea `temporada_alta` y define `X` (9 numéricas y 6 categóricas) e `y` (si la reserva se cancela).
    - Reparte train y test (80 % / 20 %, estratificado: 4800 y 1200 reservas) y define `cv`, la validación cruzada de 5 folds.
    - Define `prep` (imputar nulos y *one-hot* para las categóricas; los árboles no necesitan escalar) y produce `Xtr`, `Xte` y `nombres`.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La idea: bagging + variables al azar

#### A2 · Ejemplo — Qué reservas se quedan fuera de una muestra bootstrap

Un bosque construye muchos árboles y **promedia** sus predicciones. Para que los árboles sean *distintos entre sí* (si fueran iguales, promediar no serviría), se introduce aleatoriedad de dos formas:
1. **Bagging** (*bootstrap aggregating*): cada árbol se entrena con una **muestra con reemplazo** de las reservas (unas aparecen repetidas y otras, ≈ 37 %, no aparecen).
2. En cada pregunta, cada árbol solo puede elegir entre un **subconjunto aleatorio de variables** (`max_features`).

Cada árbol individual es peor que uno bien podado, pero **sus errores no coinciden**; al votar, el ruido se cancela y baja la varianza. Las reservas que un árbol no vio (**out-of-bag**, OOB) sirven para evaluar sin necesidad de validación cruzada.

- **Qué hace:**
    - `rng.integers(0, n, n)` hace una muestra **con reemplazo** de 5000 posiciones sobre 5000 reservas (algunas salen varias veces).
    - `set(range(n)) - set(...)` son las reservas que **no** salieron; dividido entre `n` da la proporción.
- **Qué debe salir:**
    - `Reservas que NO entran en una muestra bootstrap: 37.1%`.
- **Cómo leerlo:**
    - Es el 37 % (≈ 1/e = 36,8 %) que cada árbol no ve: esas reservas «fuera de la bolsa» (*out-of-bag*) sirven de validación gratuita.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué porcentaje de reservas no entra en una muestra bootstrap?* → **≈ 37 %** (37,1 % en esta ejecución; teóricamente 36,8 %).
2. *¿Qué quiere decir muestreo «con reemplazo»?* → Que tras sacar una reserva se **devuelve** al total: puede salir varias veces, y por eso algunas no salen ninguna.
3. *¿Para qué sirven esas reservas que quedan fuera?* → Para **evaluar** cada árbol con datos que no vio, sin reservar un conjunto aparte: es la puntuación OOB.

</details>

#### A3 · Ejercicio 1 — Bosque de 300 árboles frente a un árbol podado

- **Qué hace:**
    - `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, ...)` crea un bosque de 300 árboles, cada uno con hojas de al menos 5 reservas; `.fit(Xtr, y_train)` lo entrena.
    - `roc_auc_score(y_test, bosque.predict_proba(Xte)[:, 1])` mide su AUC en test con la probabilidad de cancelar.
    - Lo mismo con el árbol podado de la práctica anterior (`min_samples_leaf=80`) para compararlos.
- **Qué debe salir:**
    - `Árbol podado: 0.716 · Bosque de 300 árboles: 0.734`.
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `El bosque debería superar a un único árbol podado` → el árbol no tiene `min_samples_leaf=80`, o has medido el AUC con `predict` en lugar de `predict_proba`.
    - `El AUC del bosque debería superar 0,70` → no has entrenado con `Xtr, y_train` o has usado `Xte` para entrenar.
    - `NameError: name 'Xtr' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC tienen el árbol podado y el bosque?* → **0.716** el árbol y **0.734** el bosque.
2. *¿Cuánto mejora el bosque al árbol?* → **+0,018** de AUC (de 0,716 a 0,734).
3. *¿Por qué el bosque mejora a un solo árbol si cada árbol del bosque es peor que uno podado?* → Porque **sus errores no coinciden**: al votar 300 árboles distintos el ruido se cancela y baja la varianza.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2).fit(Xtr, y_train)
auc_test_bosque = roc_auc_score(y_test, bosque.predict_proba(Xte)[:, 1])
arbol = DecisionTreeClassifier(min_samples_leaf=80, random_state=SEMILLA).fit(Xtr, y_train)
auc_test_arbol = roc_auc_score(y_test, arbol.predict_proba(Xte)[:, 1])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Cuántos árboles y de qué tamaño

#### A4 · Ejemplo — El AUC out-of-bag según el número de árboles

- `n_estimators`: **más árboles casi nunca perjudica** (solo cuesta tiempo); se estabiliza.
- `min_samples_leaf` y `max_depth`: controlan el sobreajuste de cada árbol.
- `max_features`: nº de variables por pregunta (por defecto `sqrt` del total en clasificación).

Usamos la puntuación **OOB** (`oob_score=True`) para ver la evolución sin validación cruzada.

- **Qué hace:**
    - Para cada número de árboles `k` entrena un bosque con `oob_score=True`: cada reserva de train se evalúa solo con los árboles que **no** la vieron (`oob_decision_function_`).
    - Con pocos árboles, algunas reservas quedan dentro en todos y no tienen predicción OOB (NaN): `np.nan_to_num(..., nan=0.5)` las rellena con 0,5 («no sé»).
    - `roc_auc_score` calcula el AUC OOB de cada bosque y lo guarda en `oob_auc`. Dibuja el resultado (eje logarítmico). Tarda unos segundos.
- **Qué debe salir:**
    - Un gráfico: el AUC OOB sube deprisa y se aplana. `oob_auc` ≈ `[0.637, 0.676, 0.718, 0.732, 0.735, 0.738, 0.740]` para 5, 10, 25, 50, 100, 200 y 400 árboles.
- **Cómo leerlo:**
    - Pasar de 5 a 25 árboles da una subida enorme (+0,08); de 100 a 400, casi nada (+0,005): **rendimientos decrecientes**.
- **Si falla:**
    - `NameError: name 'Xtr' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC OOB tiene el bosque con 5 árboles y con 400?* → **0,637** con 5 y **0,740** con 400.
2. *¿Cuándo empieza a aplanarse la curva?* → Hacia los **50–100 árboles** (0,732 y 0,735): a partir de ahí, cada doble de árboles apenas añade milésimas.
3. *¿Qué ventaja tiene la puntuación OOB frente a la validación cruzada?* → Es **gratis**: sale del propio entrenamiento, sin repetir el ajuste 5 veces ni reservar datos aparte.

</details>

#### A5 · Ejercicio 2 — Rendimientos decrecientes

- **Qué hace:**
    - `oob_auc[-1]` es el AUC OOB con 400 árboles (el último) y `oob_auc[0]` el de 5 árboles (el primero): su resta es lo que sube (`mejora_5_a_400`).
    - `n_arboles.index(100)` da la posición del 100 en la lista (4); `oob_auc[n_arboles.index(100)]` es su AUC. La resta con el de 400 es `mejora_100_a_400`.
- **Qué debe salir:**
    - `5 → 400 árboles: +0.103 · 100 → 400 árboles: +0.005`.
    - `✔ Ejercicio 2 correcto`.
- **Si falla:**
    - `De 5 a 400 árboles el AUC debería subir bastante` → has restado al revés (primero menos último) o has usado posiciones equivocadas.
    - `De 100 a 400 la mejora es mucho menor (se satura)` → has cogido el AUC de 5 árboles en lugar del de 100.
    - `NameError: name 'oob_auc' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale `mejora_5_a_400` y cuánto `mejora_100_a_400`?* → **+0,103** y **+0,005**.
2. *¿Cuántas veces mayor es la primera mejora?* → **Unas 20 veces mayor** (0,103 / 0,005).
3. *¿Qué recomiendas: usar 100 o 400 árboles?* → Depende del coste: 4 veces más árboles son 4 veces más tiempo y memoria para ganar solo **0,005 de AUC**. Con 100–200 suele sobrar.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
mejora_5_a_400 = oob_auc[-1] - oob_auc[0]
mejora_100_a_400 = oob_auc[-1] - oob_auc[n_arboles.index(100)]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>